In [12]:
import json

file_path = "data/weather/all_combine_weather.jsonl"

data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

print(data[:1])

FileNotFoundError: [Errno 2] No such file or directory: 'data/weather/all_combine_weather.jsonl'

In [ ]:
import json
import re
import pandas as pd
import random


# TAKING 50 ROW SUBSET

In [ ]:
file_path = "data/weather/100_random_test_data.jsonl"

df = pd.read_json(file_path, lines=True)

# Sample 50 random rows
sampled_df = df.sample(n=50, random_state=42)

# Save to a new file
output_path = "data/weather/test_data_50.jsonl"
sampled_df.to_json(output_path, orient="records", lines=True)


In [ ]:

# ---------------- CONFIG ----------------
INPUT_FILE = "data/weather/new100_random_test_data.jsonl"
OUTPUT_FILE = "data/weather/100_random_test_data.jsonl"
# ----------------------------------------

OBS_BLOCK_RE = re.compile(
    r"""
    At\ (\d{4}-\d{2}-\d{2}\ \d{2}:\d{2}),\s*
    in\ (.*?),\s*the\ weather\ observations\ were\ recorded\ as\ follows:\s*
    East\ [–-]\ west\ wind\ speed\ at\ 10\ meters\ above\ ground:\s*([+\-−]?\d+(?:\.\d+)?)\ m/s,\s*
    North\ [–-]\ south\ wind\ speed\ at\ 10\ meters\ above\ ground:\s*([+\-−]?\d+(?:\.\d+)?)\ m/s\.\s*
    Dewpoint\ Temperature:\s*([+\-−]?\d+(?:\.\d+)?)\ °C\.\s*
    Temperature:\s*([+\-−]?\d+(?:\.\d+)?)\ °C\.\s*
    Pressure\ Mean\ Sea\ Level:\s*([+\-−]?\d+(?:\.\d+)?)\ hPa\.\s*
    Surface\ Pressure:\s*([+\-−]?\d+(?:\.\d+)?)\ hPa\.\s*
    Total\ Precipitation:\s*([+\-−]?\d+(?:\.\d+)?)\ meters
    """,
    re.VERBOSE | re.DOTALL | re.IGNORECASE,
)

QUESTION_LINE_RE = re.compile(
    r"please.*weather.*next.*hours",
    re.IGNORECASE
)

def float_2(x):
    return float(format(float(x), ".2f"))

def float_6(x):
    return float(format(float(x), ".4f"))

def extract_question_and_clean(obs_text: str):
    lines = obs_text.splitlines()
    question_text = None
    kept = []
    for ln in lines:
        if QUESTION_LINE_RE.search(ln) and question_text is None:
            question_text = ln.strip()
        else:
            kept.append(ln)
    return question_text, "\n".join(kept)

def parse_observation(obs_text: str, question_text=None, area=None, lat=None, lon=None):
    results = {}
    for m in OBS_BLOCK_RE.finditer(obs_text):
        valid_time_min, area_text, u10, v10, d2m, t2m, msl, sp, tp = m.groups()
        valid_time = f"{valid_time_min}:00"
        results[valid_time] = {
            "east_west_wind_speed_10m": float_2(u10),
            "north_south_wind_speed_10m": float_2(v10),
            "dewpoint_temperature_2m": float_2(d2m),
            "air_temperature_2m": float_2(t2m),
            "mean_sea_level_pressure": float_2(msl),
            "surface_pressure": float_2(sp),
            "total_precipitation": float_6(tp),
            "latitude": round(lat, 2) if lat is not None else None,
            "longitude": round(lon, 2) if lon is not None else None,
            "area": area_text or area,
        }
    if question_text:
        results["question"] = question_text
    return results

def list_to_dict_by_time(gt_list):
    """ubah ground_truth list of dicts -> dict keyed by valid_time + rename keys"""
    if not isinstance(gt_list, list):
        return gt_list
    result = {}
    for item in gt_list:
        vt = item.get("valid_time")
        if not vt:
            continue
        result[vt] = {
            "east_west_wind_speed_10m": round(item.get("u10", 0.0), 2),
            "north_south_wind_speed_10m": round(item.get("v10", 0.0), 2),
            "dewpoint_temperature_2m": round(item.get("d2m", 0.0), 2),
            "air_temperature_2m": round(item.get("t2m", 0.0), 2),
            "mean_sea_level_pressure": round(item.get("msl", 0.0), 2),
            "surface_pressure": round(item.get("sp", 0.0), 2),
            "total_precipitation": float_6(item.get("tp", 0.0)),
            "latitude": round(item.get("latitude", 0.0), 2) if "latitude" in item else None,
            "longitude": round(item.get("longitude", 0.0), 2) if "longitude" in item else None,
            "area": item.get("area")
        }
    return result

def process_jsonl(in_path: str, out_path: str):
    total = parsed = 0
    with open(in_path, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            total += 1
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                fout.write(line + "\n")
                continue

            obs_text = obj.get("observation")
            gt = obj.get("ground_truth")

            # convert ground_truth
            obj["ground_truth"] = list_to_dict_by_time(gt)

            if isinstance(obs_text, str):
                question_text, obs_core = extract_question_and_clean(obs_text)
                obs_dict = parse_observation(obs_core, question_text=question_text)
                obj["obs_json"] = obs_dict
                if obs_dict:
                    parsed += 1
            else:
                obj["obs_json"] = {}

            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

    print(f"Done. Processed {total} lines; obs_json parsed for {parsed} lines.")

if __name__ == "__main__":
    process_jsonl(INPUT_FILE, OUTPUT_FILE)



In [ ]:
import json
from typing import Dict, Any

# ==== KONFIGURASI I/O ====
INPUT_FILE = "data/weather/new100_random_test_data.jsonl"             # ganti sesuai kebutuhan
OUTPUT_FILE = "data/weather/100_test_data.jsonl"    # file keluaran

# ==== MAPPING KEY ====
KEY_MAP = {
    "u10": "east_west_wind_speed_10m",
    "v10": "north_south_wind_speed_10m",
    "d2m": "dewpoint_temperature_2m",
    "t2m": "air_temperature_2m",
    "msl": "mean_sea_level_pressure",
    "sp": "surface_pressure",
    "tp": "total_precipitation",
    "latitude": "latitude",
    "longitude": "longitude",
    "area": "area",
}

# key yang harus 6 desimal; lainnya 2 desimal
TP_KEYS = {"tp", "total_precipitation"}
DEFAULT_PRECISION = 2
TP_PRECISION = 6


def _round_number(value: Any, precision: int) -> Any:
    """Bulatkan angka; selain int/float dikembalikan apa adanya."""
    if isinstance(value, (int, float)):
        return round(float(value), precision)
    return value


def _rename_and_round_record(record: Dict[str, Any]) -> Dict[str, Any]:
    """
    Rename key dan round angka untuk satu blok (mis. ground_truth atau obs_json)
    yang strukturnya: { timestamp: { var_key: value, ... }, ... }
    """
    out: Dict[str, Any] = {}
    for ts, payload in record.items():
        if not isinstance(payload, dict):
            # kalau formatnya tidak seperti yang diharapkan, salin apa adanya
            out[ts] = payload
            continue

        new_payload: Dict[str, Any] = {}
        for old_key, value in payload.items():
            new_key = KEY_MAP.get(old_key, old_key)

            # tentukan presisi
            if new_key in TP_KEYS or old_key in TP_KEYS:
                precision = TP_PRECISION
            else:
                precision = DEFAULT_PRECISION

            # bulatkan angka jika numeric; biarkan None/str/dll apa adanya
            new_payload[new_key] = _round_number(value, precision)

        out[ts] = new_payload
    return out


def process_jsonl(input_path: str, output_path: str) -> None:
    """
    Baca JSONL baris per baris, rename+round untuk ground_truth dan obs_json (jika ada),
    lalu tulis ke file output.
    """
    total = 0
    with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
        for line_num, line in enumerate(fin, 1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"[Baris {line_num}] JSON tidak valid: {e}")
                continue

            # ground_truth
            if isinstance(obj.get("ground_truth"), dict):
                obj["ground_truth"] = _rename_and_round_record(obj["ground_truth"])

            # obs_json
            if isinstance(obj.get("obs_json"), dict):
                obj["obs_json"] = _rename_and_round_record(obj["obs_json"])

            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
            total += 1

    print(f"Selesai. {total} baris diproses → {output_path}")


if __name__ == "__main__":
    process_jsonl(INPUT_FILE, OUTPUT_FILE)


In [2]:
import json

file_path = "data/weather/10_test_data.jsonl"

data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

print("Items \n", data[1].keys())
print("Observation \n", data[7]['observation'])
print("Result \n", data[7]['result'])
print("Ground Truth \n", data[7]['ground_truth'])
print("Obs in Json  \n", data[7]['obs_json'])

Items 
 dict_keys(['observation', 'result', 'ground_truth', 'obs_json'])
Observation 
 At 2014-08-01 08:00, in Yalimo, Highland Papua, the weather observations were recorded as follows:
East – west wind speed at 10 meters above ground: -0.87 m/s, North – south wind speed at 10 meters above ground: 0.32 m/s. Dewpoint Temperature: 24.33 °C. Temperature: 28.16 °C. Pressure Mean Sea Level: 1009.16 hPa. Surface Pressure: 961.61 hPa. Total Precipitation: 0.0000 meters
At 2014-08-01 09:00, in Yalimo, Highland Papua, the weather observations were recorded as follows:
East – west wind speed at 10 meters above ground: -0.68 m/s, North – south wind speed at 10 meters above ground: 0.58 m/s. Dewpoint Temperature: 22.76 °C. Temperature: 26.50 °C. Pressure Mean Sea Level: 1009.48 hPa. Surface Pressure: 961.88 hPa. Total Precipitation: 0.0000 meters
At 2014-08-01 10:00, in Yalimo, Highland Papua, the weather observations were recorded as follows:
East – west wind speed at 10 meters above ground: -0.5

In [2]:
import json

file_path = "data/weather_extreme/10_dataset.jsonl"

data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

print(data[1].keys())

print("Items \n", data[3].keys())
print("Observation \n", data[3]['observation'])
print("Question \n", data[3]['question'])
print("Ground Truth \n", data[3]['ground_truth'])


dict_keys(['observation', 'result', 'obs_json', 'ground_truth', 'question'])
Items 
 dict_keys(['observation', 'result', 'obs_json', 'ground_truth', 'question'])
Observation 
 At 2023-06-02 16:00:00, at the location (32.0, -102.0), the weather observations were recorded as follows: Eastward wind speed at 10 meters above ground: -2.176 m/s, Northward wind speed at 10 meters above ground: 7.498 m/s, Eastward wind speed at 100 meters above ground: -2.813 m/s, Northward wind speed at 100 meters above ground: 9.939 m/s, Mean sea level pressure: 1010.846 hPa, Instantaneous 10 meters wind gust: 12.845 m/s, 2 meters temperature: 24.963 C, 2 meters dewpoint temperature: 18.605 C, Total precipitation: 0.000 m, Surface pressure: 917.341 hPa. At 2023-06-02 17:00:00, at the location (32.0, -102.0), the weather observations were recorded as follows: Eastward wind speed at 10 meters above ground: -2.434 m/s, Northward wind speed at 10 meters above ground: 7.405 m/s, Eastward wind speed at 100 meters 

In [7]:
file_path = "data/weather/100_test_data.jsonl"

data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

print(data[1].keys())

print("Items \n", data[1].keys())
print("result \n", data[1]['result'])
print("ground_truth \n", data[1]['ground_truth'])
print("obs_json \n", data[1]['obs_json'])

dict_keys(['observation', 'result', 'ground_truth', 'obs_json'])
Items 
 dict_keys(['observation', 'result', 'ground_truth', 'obs_json'])
result 
 At 2010-11-05 23:00, in Cisarua, West Java, the weather observations were recorded as follows: East – west wind speed at 10 meters above ground: 0.87 m/s, North – south wind speed at 10 meters above ground: 2.24 m/s. Dewpoint Temperature: 20.64 °C. Temperature: 21.99 °C. Pressure Mean Sea Level: 1010.58 hPa. Surface Pressure: 946.82 hPa. Total Precipitation: 0.0000 meters At 2010-11-06 00:00, in Cisarua, West Java, the weather observations were recorded as follows: East – west wind speed at 10 meters above ground: 0.73 m/s, North – south wind speed at 10 meters above ground: 1.88 m/s. Dewpoint Temperature: 20.45 °C. Temperature: 22.19 °C. Pressure Mean Sea Level: 1011.11 hPa. Surface Pressure: 947.30 hPa. Total Precipitation: 0.0000 meters At 2010-11-06 01:00, in Cisarua, West Java, the weather observations were recorded as follows: East – w

In [1]:
from experiments.kfold_extreme import load_jsonl, print_class_distribution
data = load_jsonl("data/extreme_weather/100_dataset.jsonl")
print_class_distribution(data, label="100-sample dataset")

ModuleNotFoundError: No module named 'pandas'